# Model: will the runner be safe?

## Setup

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

TEST_SEASON = 2026
FEATURES = ["launch_speed", "launch_angle", "spray_angle", "outs_when_up",
            "runner_speed", "fielder_arm", "bb_type"]

## Load the plays

In [ ]:
plays = pd.read_csv("../opportunities.csv")
plays = plays[plays.complete]

X = pd.get_dummies(plays[FEATURES], columns=["bb_type"], dtype=float)

train = plays.sent & (plays.season < TEST_SEASON)
test = plays.season == TEST_SEASON

## Train the model and predict 2026

I chose a *gradient boosting* model. It builds a long chain of small decision trees. Each tree is a couple of yes/no questions, like "Was the ball hit harder than 95 mph?" or "Is the runner faster than 28 ft/s?", that end in a guess. Each new tree focuses on fixing the mistakes of the trees before it, and the final answer combines all of them.

I set `max_depth=2` to limit each tree to two questions in a row. Outs are rare, only about 350 in four seasons, so bigger trees would start memorizing flukes instead of learning real patterns.

I set `early_stopping=False` so the model learns from every training send. Without it, scikit-learn quietly sets aside 10% of the plays whenever there are more than 10,000. I left everything else, including the number of trees (100), at scikit-learn's standard settings.

`predict_proba` gives two chances for each play, out and safe. I keep `[:, 1]`, the chance of safe.

In [ ]:
model = HistGradientBoostingClassifier(max_depth=2, early_stopping=False, random_state=0)
model.fit(X[train], plays.safe[train].astype(int))
plays.loc[test, "p_safe"] = model.predict_proba(X[test])[:, 1]

## Save

In [ ]:
plays.to_csv("../scored.csv", index=False)